# exp158 segment continuity selector on exp157

exp157 の saved booster、exp099 v2 cache、exp072 dense cache を使って OOF score surface を復元し、well-local Viterbi DP で candidate path を平滑化する train-side audit。

## Contents

1. Setup and configuration
2. Input artifact checks
3. Reconstruct exp157 scores and run continuity selector
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json

import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from segment_continuity_selector_on_exp157 import run_segment_continuity_selector

config = load_config()
paths = ExperimentPaths()
paths.ensure_output_dirs()

artifact_dir = paths.artifacts_dir
print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('parent:', get_nested(config, 'lineage.parent'))
print('cache_parent:', get_nested(config, 'lineage.cache_parent'))
print('default candidate:', get_nested(config, 'selector.default_candidate'))
print('switch candidates:', get_nested(config, 'selector.allowed_switch_candidates'))
grid = get_nested(config, 'selector.viterbi_grid')
variant_count = 1
for key, value in grid.items():
    if isinstance(value, list):
        variant_count *= len(value)
print('viterbi variant count:', variant_count)
print('new boosters:', get_nested(config, 'model.trains_new_boosters'))

## 2. Input artifact checks

In [ ]:
input_paths = {
    'exp099_train_feature_cache_local': get_nested(config, 'data.exp099_train_feature_cache_local'),
    'exp099_train_feature_schema_local': get_nested(config, 'data.exp099_train_feature_schema_local'),
    'exp072_train_feature_cache_local': get_nested(config, 'data.exp072_train_feature_cache_local'),
    'exp072_feature_schema_local': get_nested(config, 'data.exp072_feature_schema_local'),
    'exp157_artifact_dir_local': get_nested(config, 'data.exp157_artifact_dir_local'),
    'exp157_model_manifest': get_nested(config, 'data.exp157_model_manifest'),
    'exp157_feature_schema': get_nested(config, 'data.exp157_feature_schema'),
}
for name, value in input_paths.items():
    print(f'{name}: {value}')

print('artifact_dir:', artifact_dir)
print('expected outputs:')
for item in get_nested(config, 'audit.expected_train_artifacts'):
    print(' -', item)

## 3. Reconstruct exp157 scores and run continuity selector

In [ ]:
summary = run_segment_continuity_selector(
    output_dir=artifact_dir,
    cache_path=None,
    schema_path=None,
    max_rows=get_nested(config, 'selector.max_rows'),
)
print(json.dumps(summary, indent=2, sort_keys=True)[:4000])

## 4. Metrics and artifacts

In [ ]:
prefix = get_nested(config, 'audit.output_prefix')
metrics_path = artifact_dir / f'{prefix}_metrics.csv'
by_well_path = artifact_dir / f'{prefix}_by_well.csv'
bucket_path = artifact_dir / f'{prefix}_bucket_metrics.csv'
distribution_path = artifact_dir / f'{prefix}_selection_distribution.csv'
params_path = artifact_dir / f'{prefix}_viterbi_params.csv'
summary_path = artifact_dir / f'{prefix}_summary.json'

metrics = pd.read_csv(metrics_path)
display(metrics.head(20))
display(pd.read_csv(distribution_path).head(30))
display(pd.read_csv(by_well_path).head(20))
display(pd.read_csv(bucket_path).head(30))
display(pd.read_csv(params_path).head(20))

with summary_path.open() as fp:
    saved_summary = json.load(fp)
print('summary:', summary_path)
print('recommendation:', saved_summary.get('recommendation'))
print('best_viterbi_variant:', saved_summary.get('best_viterbi_variant'))
print('delta_rmse_vs_likpf_mean:', saved_summary.get('delta_rmse_vs_likpf_mean'))